In [20]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import numpy as np


In [2]:
data = pd.read_csv(r'C:\Users\ebrah\NLP\Corona.csv', encoding='latin1')
data.head()

,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
0,3799,48751,London,16-03-2020,@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...,Neutral
1,3800,48752,UK,16-03-2020,advice Talk to your neighbours family to excha...,Positive
2,3801,48753,Vagabonds,16-03-2020,Coronavirus Australia: Woolworths to give elde...,Positive
3,3802,48754,NaN,16-03-2020,My food stock is not the only one which is emp...,Positive
4,3803,48755,NaN,16-03-2020,"Me, ready to go at supermarket during the #COV...",Extremely Negative


In [3]:
data.drop(['UserName', 'ScreenName'], axis=1, inplace=True)

In [4]:
data['Sentiment'].value_counts()

Sentiment
Positive              11422
Negative               9917
Neutral                7713
Extremely Positive     6624
Extremely Negative     5481
Name: count, dtype: int64

In [5]:
data['OriginalTweet'].head(10).to_list()

['@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/iFz9FAn2Pa and https://t.co/xX6ghGFzCC and https://t.co/I2NlzdxNo8',
 'advice Talk to your neighbours family to exchange phone numbers create contact list with phone numbers of neighbours schools employer chemist GP set up online shopping accounts if poss adequate supplies of regular meds but not over order',
 'Coronavirus Australia: Woolworths to give elderly, disabled dedicated shopping hours amid COVID-19 outbreak https://t.co/bInCA9Vp8P',
 "My food stock is not the only one which is empty...\r\r\n\r\r\nPLEASE, don't panic, THERE WILL BE ENOUGH FOOD FOR EVERYONE if you do not take more than you need. \r\r\nStay calm, stay safe.\r\r\n\r\r\n#COVID19france #COVID_19 #COVID19 #coronavirus #confinement #Confinementotal #ConfinementGeneral https://t.co/zrlG0Z520j",
 "Me, ready to go at supermarket during the #COVID19 outbreak.\r\r\n\r\r\nNot because I'm paranoid, but because my food stock is litteraly empty. The #coronavirus is a serious thin

In [6]:
data['CleanTweet'] = data['OriginalTweet'].str.lower()

In [7]:
data['CleanTweet'] = data['CleanTweet'].apply(lambda x : re.sub(r'http\S+|www\S+|https\S+', '', x))

In [8]:
data['CleanTweet'] = data['CleanTweet'].apply(lambda x : re.sub(r'@\w+', '', x))

In [9]:
data['CleanTweet'] = data['CleanTweet'].str.replace('#', '', regex=False)

In [10]:
data['CleanTweet'] = data['CleanTweet'].apply(lambda x: re.sub(r'[^a-zA-Z\s]', '', x))

In [11]:
data['CleanTweet'] = data['CleanTweet'].apply(lambda x: re.sub(r'\s+', ' ', x).strip())

In [12]:
data[['OriginalTweet', 'CleanTweet']]

,OriginalTweet,CleanTweet
0,@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...,and and
1,advice Talk to your neighbours family to excha...,advice talk to your neighbours family to excha...
2,Coronavirus Australia: Woolworths to give elde...,coronavirus australia woolworths to give elder...
3,My food stock is not the only one which is emp...,my food stock is not the only one which is emp...
4,"Me, ready to go at supermarket during the #COV...",me ready to go at supermarket during the covid...
...,...,...
41152,Airline pilots offering to stock supermarket s...,airline pilots offering to stock supermarket s...
41153,Response to complaint not provided citing COVI...,response to complaint not provided citing covi...
41154,You know itÂs getting tough when @KameronWild...,you know its getting tough when is rationing t...
41155,Is it wrong that the smell of hand sanitizer i...,is it wrong that the smell of hand sanitizer i...


In [13]:
data['CleanTweetLen'] = data['CleanTweet'].apply(len)

In [14]:
data['TweetAt'] = pd.to_datetime(data['TweetAt'])

data['Year'] = data['TweetAt'].dt.year
data['Month'] = data['TweetAt'].dt.month
data['Day'] = data['TweetAt'].dt.day
data['Dayofweek'] = data['TweetAt'].dt.dayofweek


C:\Users\ebrah\AppData\Local\Temp\ipykernel_17420\3206437102.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  data['TweetAt'] = pd.to_datetime(data['TweetAt'])


In [15]:
data.head()

,Location,TweetAt,OriginalTweet,Sentiment,CleanTweet,CleanTweetLen,Year,Month,Day,Dayofweek
0,London,2020-03-16,@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...,Neutral,and and,7,2020,3,16,0
1,UK,2020-03-16,advice Talk to your neighbours family to excha...,Positive,advice talk to your neighbours family to excha...,237,2020,3,16,0
2,Vagabonds,2020-03-16,Coronavirus Australia: Woolworths to give elde...,Positive,coronavirus australia woolworths to give elder...,102,2020,3,16,0
3,NaN,2020-03-16,My food stock is not the only one which is emp...,Positive,my food stock is not the only one which is emp...,246,2020,3,16,0
4,NaN,2020-03-16,"Me, ready to go at supermarket during the #COV...",Extremely Negative,me ready to go at supermarket during the covid...,256,2020,3,16,0


In [18]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(data['Sentiment'])

X = data[['OriginalTweet', 'Month', 'Day', 'Dayofweek']]

In [22]:
text_data = data['CleanTweet']

# 2️⃣ TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=3000, stop_words='english', ngram_range=(1,2))
X_text = tfidf.fit_transform(text_data).toarray()

# 3️⃣ Add numeric features
X_numeric = data[['Month', 'Day', 'Dayofweek']].values

# Combine text + numeric features
X = np.hstack([X_text, X_numeric])

# 4️⃣ Encode target
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(data['Sentiment'])

# 5️⃣ Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6️⃣ Train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# 7️⃣ Predict and evaluate
y_pred = model.predict(X_test)

print("\n✅ Accuracy:", accuracy_score(y_test, y_pred))
print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))


✅ Accuracy: 0.5539358600583091

📊 Classification Report:
                    precision    recall  f1-score   support

Extremely Negative       0.62      0.47      0.53      1056
Extremely Positive       0.67      0.55      0.60      1330
          Negative       0.49      0.49      0.49      2006
           Neutral       0.59      0.66      0.62      1553
          Positive       0.51      0.58      0.54      2287

          accuracy                           0.55      8232
         macro avg       0.58      0.55      0.56      8232
      weighted avg       0.56      0.55      0.55      8232



C:\Users\ebrah\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
